# VCAT: Load and Explore Voynich EVA Data

This notebook demonstrates how to load the Voynich EVA transcription dataset from Hugging Face.

In [ ]:
import sys
from pathlib import Path

# The notebooks run from notebooks/; the project root is its parent
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root.name}")

In [ ]:
from datasets import load_dataset

# The local build (python -m builders.build_eva_lines). The Hugging Face copy, Ched-ai/voynich-eva, still
# holds the 0.2.3 rows until the next release (see the last cell).
ds = load_dataset("parquet", data_files=str(project_root / "output" / "eva_lines.parquet"))
print(ds)

In [ ]:
# View first record
print(ds["train"][0])

In [ ]:
# Convert to pandas for easier exploration

df = ds["train"].to_pandas()
print(f"Total records: {len(df)}")
df.head()

## Basic Statistics

In [ ]:
# Count by Currier language
print("Lines by Currier Language:")
print(df["currier_language"].value_counts())

In [ ]:
# Count by section
print("\nLines by Section:")
print(df["section"].value_counts())

In [ ]:
# Count by line type
print("\nLines by Type:")
print(df["line_type"].value_counts())

## Filter Examples

In [ ]:
# Get all herbal section lines
herbal = df[df["section"].fillna("").str.startswith("herbal")]
print(f"Herbal section: {len(herbal)} lines")

In [ ]:
# Get lines with uncertain readings
uncertain = df[df["has_uncertain"]]
print(f"Lines with uncertain readings: {len(uncertain)}")

In [ ]:
# Compare Language A vs Language B
lang_a = df[df["currier_language"] == "A"]
lang_b = df[df["currier_language"] == "B"]

print(f"Language A: {len(lang_a)} lines, avg words: {lang_a['word_count'].mean():.1f}")
print(f"Language B: {len(lang_b)} lines, avg words: {lang_b['word_count'].mean():.1f}")

## Character Frequency Analysis

In [ ]:
from collections import Counter

# Count all characters in clean text
all_text = "".join(df["text_clean"].fillna(""))
# Remove separators
all_text = all_text.replace(".", "").replace(",", "")

char_counts = Counter(all_text)
print("Top 15 characters:")
for char, count in char_counts.most_common(15):
    pct = count / len(all_text) * 100
    print(f"  {char}: {count:,} ({pct:.1f}%)")

## Local Loading (from Parquet file)

If you have the local parquet file from the repository:

In [ ]:
# From the Hugging Face Hub instead (network; the published copy is 0.2.3 until the next release):
# ds = load_dataset("Ched-ai/voynich-eva", "lines")